In [2]:
!pip install monai nibabel --quiet
import torch
import nibabel as nib
import numpy as np
import zipfile
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from monai.networks.nets import AttentionUnet
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose, ResizeWithPadOrCropd,
    ScaleIntensityRangePercentilesd,
    RandFlipd, RandRotate90d, RandAffined,
)
print("GPU available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.0 MB/s eta 0:00:00
GPU available: True


In [3]:
from google.colab import drive
drive.mount("/content/drive")

zip_path     = Path("/content/drive/MyDrive/Labeled.zip")
extract_path = Path("/content/Labeled")

print("Extracting zip...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

labeled_path = extract_path / "Labeled"
patients = [p for p in sorted(labeled_path.iterdir()) if p.is_dir()]
print(f"Found {len(patients)} patient folders")

Mounted at /content/drive
Extracting zip...
Found 150 patient folders


In [5]:
slices_dir = Path("/content/slices")
slices_dir.mkdir(exist_ok=True)

slice_dicts   = []
skipped_empty = 0
patients_ok   = 0

for patient_dir in sorted(labeled_path.iterdir()):
    if not patient_dir.is_dir():
        continue

    img_files = list(patient_dir.glob("*_sa.nii.gz"))
    gt_files  = list(patient_dir.glob("*_sa_gt.nii.gz"))
    if not img_files or not gt_files:
        continue

    img_vol = np.asarray(nib.load(str(img_files[0])).dataobj, dtype=np.float32)
    lbl_vol = np.asarray(nib.load(str(gt_files[0])).dataobj,  dtype=np.int64)
    pid     = patient_dir.name

    n_slices = img_vol.shape[2]
    n_frames = img_vol.shape[3] if img_vol.ndim == 4 else 1

    for t in range(n_frames):
        frame_lbl = lbl_vol[..., t] if lbl_vol.ndim == 4 else lbl_vol
        if frame_lbl.max() == 0:
            skipped_empty += n_slices
            continue

        for s in range(n_slices):
            img_2d = img_vol[..., s, t] if img_vol.ndim == 4 else img_vol[..., s]
            lbl_2d = lbl_vol[..., s, t] if lbl_vol.ndim == 4 else lbl_vol[..., s]

            img_path = slices_dir / f"{pid}_s{s}_t{t}_img.npy"
            lbl_path = slices_dir / f"{pid}_s{s}_t{t}_lbl.npy"

            if not img_path.exists():
                np.save(img_path, img_2d)
                np.save(lbl_path, lbl_2d)

            slice_dicts.append({
                "image": str(img_path),
                "label": str(lbl_path)
            })

    patients_ok += 1

print(f"Patients processed   : {patients_ok}")
print(f"Valid ED/ES samples  : {len(slice_dicts)}")
print(f"Skipped empty slices : {skipped_empty}")

train_dicts, val_dicts = train_test_split(
    slice_dicts, test_size=0.2, random_state=42
)
print(f"Train: {len(train_dicts)}  Val: {len(val_dicts)}")

Patients processed   : 150
Valid ED/ES samples  : 3286
Skipped empty slices : 40194
Train: 2628  Val: 658


In [6]:
class NpySliceDataset(Dataset):
    def __init__(self, slice_dicts, transform=None):
        self.data      = slice_dicts
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d     = self.data[idx]
        img2d = np.load(d["image"])[None].astype(np.float32)
        lbl2d = np.load(d["label"])[None].astype(np.int64)

        if self.transform:
            sample = self.transform({"image": img2d, "label": lbl2d})
            img2d  = sample["image"]
            lbl2d  = sample["label"]

        if isinstance(img2d, torch.Tensor):
            return img2d.float(), lbl2d.long()
        return torch.from_numpy(img2d).float(), torch.from_numpy(lbl2d).long()


train_transforms = Compose([
    ResizeWithPadOrCropd(keys=["image","label"], spatial_size=(256, 256)),
    ScaleIntensityRangePercentilesd(
        keys=["image"], lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True
    ),
    RandFlipd(keys=["image","label"],    prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image","label"],    prob=0.5, spatial_axis=1),
    RandRotate90d(keys=["image","label"], prob=0.5, max_k=3),
    RandAffined(
        keys=["image","label"], prob=0.3,
        rotate_range=(0.2,), scale_range=(0.1,),
        mode=("bilinear","nearest"), padding_mode="zeros"
    ),
])

val_transforms = Compose([
    ResizeWithPadOrCropd(keys=["image","label"], spatial_size=(256, 256)),
    ScaleIntensityRangePercentilesd(
        keys=["image"], lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True
    ),
])

train_ds = NpySliceDataset(train_dicts, transform=train_transforms)
val_ds   = NpySliceDataset(val_dicts,   transform=val_transforms)

train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")

Train batches: 165
Val batches  : 42


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = AttentionUnet(
    spatial_dims=2,
    in_channels=1,
    out_channels=4,
    channels=(32, 64, 128, 256, 320),
    strides=(2, 2, 2, 2),
).to(device)

criterion = DiceFocalLoss(
    to_onehot_y=True,
    softmax=True,
    gamma=2.0,
    lambda_dice=0.5,
    lambda_focal=0.5,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2
)
dice_metric = DiceMetric(include_background=False, reduction="none")
scaler      = torch.cuda.amp.GradScaler()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"AttentionUNet — {total_params/1e6:.1f}M trainable parameters")

Device: cuda
AttentionUNet — 5.6M trainable parameters


/tmp/ipykernel_851/2569376344.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler      = torch.cuda.amp.GradScaler()


In [10]:
best_val   = 0.0
patience   = 10
counter    = 0
NUM_EPOCHS = 60

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()

    model.eval()
    dice_metric.reset()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda"):
                outputs   = model(images)
                val_loss += criterion(outputs, labels).item()

            preds = torch.argmax(outputs, dim=1, keepdim=True).long()
            dice_metric(y_pred=preds, y=labels)
    all_dice  = dice_metric.aggregate()
    per_class = torch.nanmean(all_dice, dim=0)
    def safe_dice(tensor, idx):
        """Returns dice for class idx, or 0.0 if index missing or NaN."""
        if idx >= len(tensor):
            return 0.0
        val = tensor[idx].item()
        return 0.0 if (val != val) else val
    lv_dice   = safe_dice(per_class, 0)
    myo_dice  = safe_dice(per_class, 1)
    rv_dice   = safe_dice(per_class, 2)
    mean_dice = np.nanmean([lv_dice, myo_dice, rv_dice])

    print(
        f"Epoch {epoch+1:3d} | "
        f"Loss: {train_loss/len(train_loader):.4f} | "
        f"LV: {lv_dice:.4f}  "
        f"Myo: {myo_dice:.4f}  "
        f"RV: {rv_dice:.4f} | "
        f"Mean: {mean_dice:.4f}"
    )

    if mean_dice > best_val:
        best_val = mean_dice
        counter  = 0
        torch.save({
            "epoch":           epoch + 1,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_dice":        mean_dice,
            "lv_dice":         lv_dice,
            "myo_dice":        myo_dice,
            "rv_dice":         rv_dice,
        }, "best_attention_unet.pth")
        print("  ↑ checkpoint saved")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

print(f"\nTraining complete Best Mean Dice: {best_val:.4f}")

Epoch   1 | Loss: 0.4634 | LV: 0.4291  Myo: 0.0000  RV: 0.0000 | Mean: 0.1430
  ↑ checkpoint saved
Epoch   2 | Loss: 0.3854 | LV: 0.8120  Myo: 0.0000  RV: 0.0000 | Mean: 0.2707
  ↑ checkpoint saved
Epoch   3 | Loss: 0.3167 | LV: 0.8455  Myo: 0.0000  RV: 0.0000 | Mean: 0.2818
  ↑ checkpoint saved
Epoch   4 | Loss: 0.2629 | LV: 0.8652  Myo: 0.0000  RV: 0.0000 | Mean: 0.2884
  ↑ checkpoint saved
Epoch   5 | Loss: 0.2265 | LV: 0.8832  Myo: 0.0000  RV: 0.0000 | Mean: 0.2944
  ↑ checkpoint saved
Epoch   6 | Loss: 0.2047 | LV: 0.8922  Myo: 0.0000  RV: 0.0000 | Mean: 0.2974
  ↑ checkpoint saved
Epoch   7 | Loss: 0.1921 | LV: 0.8882  Myo: 0.0000  RV: 0.0000 | Mean: 0.2961
Epoch   8 | Loss: 0.1842 | LV: 0.9055  Myo: 0.0000  RV: 0.0000 | Mean: 0.3018
  ↑ checkpoint saved
Epoch   9 | Loss: 0.1792 | LV: 0.9121  Myo: 0.0000  RV: 0.0000 | Mean: 0.3040
  ↑ checkpoint saved
Epoch  10 | Loss: 0.1774 | LV: 0.9105  Myo: 0.0000  RV: 0.0000 | Mean: 0.3035
Epoch  11 | Loss: 0.1804 | LV: 0.8843  Myo: 0.0000  

In [21]:
%cd /content/cardiac-mri-ef-calculation

!cp "/content/drive/MyDrive/Colab Notebooks/Cardiac_MRI.ipynb" .

/content/cardiac-mri-ef-calculation
cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/Cardiac_MRI.ipynb': No such file or directory


In [22]:
!git reset --soft HEAD~1

In [23]:
!git add Cardiac_MRI.ipynb

fatal: pathspec 'Cardiac_MRI.ipynb' did not match any files


In [24]:
%cd /content/cardiac-mri-ef-calculation

/content/cardiac-mri-ef-calculation


In [25]:
!ls -lh *.ipynb

-rw------- 1 root root 36K Jul  2 16:30 MRI.ipynb
